##Homework 3 Question 2

In [1]:
!pip install syncode

!python --version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.1/206.1 kB 7.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.1 MB/s eta 0:00:00
  Created wheel for syncode: filename=syncode-0.4.16-py3-none-any.whl size=237009 sha256=e0fbedc06b226a15ed59b794b223f07830a43caa8d3083f5c5f1d1e10792f261
  Stored in directory: /root/.cache/pip/wheels/3f/a5/9b/9f6e1723cd27ffd3

In [2]:
from syncode import Syncode
import warnings
warnings.filterwarnings('ignore')

grammar = """
start: expr

?expr: term
     | expr "+" term  -> add
     | expr "-" term  -> subtract

?term: factor
     | term "*" factor  -> multiply
     | term "/" factor  -> divide

?factor: NUMBER        -> number
       | "(" expr ")"

%import common.NUMBER
%import common.WS
%ignore WS
"""

# Few-shot examples I came up with
fewshot_prompt = """Extract the two numbers and the operator from each math question.
Use + for plus, - for minus, * for multiply, and / for division.
Write every number as a float.

Question: What is 56. plus 7.25?
Extraction: 56.0 + 7.25

Question: What is 98.6 minus 12.4?
Extraction: 98.6 - 12.4

Question: What is 3.5 multiplied by 20.?
Extraction: 3.5 * 20.0

Question: What is 81.9 divided by 9.0?
Extraction: 81.9 / 9.0

"""

from syncode.larkm import Lark

calc_parser = Lark(grammar, parser="lalr")

OPERATOR_BY_RULE_NAME = {"add": "+", "subtract": "-", "multiply": "*", "divide": "/"}

In [3]:
model_name = "microsoft/phi-2"
syn_llm = Syncode(model = model_name, grammar=grammar, parse_output_only=True, max_new_tokens=20)

[2026-10-27 02:20:15,462-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

[2026-10-27 02:21:46,745-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_8407133080_50257.pkl exist: False
[2026-10-27 02:21:46,746-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_8407133080_50257.pkl.
[2026-10-27 02:21:47,642-syncode.mask_store.fsm_set] - 8 FSMs with 36 states initialized in 0.02 seconds
[2026-10-27 02:21:47,644-syncode.mask_store.mask_store] - Ignore whitespace is True
[2026-10-27 02:21:47,647-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 64 to 28


100%|██████████| 36/36 [00:10<00:00,  3.54it/s]

[2026-10-27 02:21:57,882-syncode.mask_store.mask_store] - Time taken to create mask store: 11.09 seconds


In [4]:
def calculate(n1, operator, n2):
    if operator == "+":
        res = n1 + n2
    elif operator == "-":
        res = n1 - n2
    elif operator == "*":
        res = n1 * n2
    elif operator == "/":
        if n2 == 0:
            return float("nan")
        res = n1 / n2
    else:
        return float("nan")

    return float(round(res, 10))


def run_syncode(textual_query):
    #extract
    prompt = fewshot_prompt + "Question: " + textual_query.strip() + "\nExtraction:"
    syncode_output = syn_llm.infer(prompt, stop_words=["\n"])[0]

    # parse the output with the calc grammar
    try:
        tree = calc_parser.parse(syncode_output)
    except Exception:
        print("Can't parse the SynCode output:", repr(syncode_output))
        return ([], "", float("nan"))

    operation = tree.children[0]
    if operation.data not in OPERATOR_BY_RULE_NAME:
        print("SynCode output is not one operation:", repr(syncode_output))
        return ([], "", float("nan"))

    l,r = operation.children
    if l.data != "number" or r.data != "number":
        print("SynCode output has more than two numbers:", repr(syncode_output))
        return ([], "", float("nan"))

    n1 = float(l.children[0])
    operator = OPERATOR_BY_RULE_NAME[operation.data]
    n2 = float(r.children[0])
    final_answer = calculate(n1, operator, n2)

    operands = [n1, n2]
    return (operands, operator, final_answer)

In [5]:
#Test with the examples from the homework, I added a minus one too
queries = [
    "What is 327. multiplied by 11.0?",
    "What is 45.1 plus 23.54?",
    "What is 120.4 divided by 4.0?",
    "What is 10.5 minus 2.25?",
]

for query in queries:
    operands, operator, final_answer = run_syncode(query)
    print(f"Query: {query}")
    print(f"Operands: {operands}  Operator: {operator}  Final Answer: {final_answer}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Query: What is 327. multiplied by 11.0?
Operands: [327.0, 11.0]  Operator: *  Final Answer: 3597.0
Query: What is 45.1 plus 23.54?
Operands: [45.1, 23.54]  Operator: +  Final Answer: 68.64
Query: What is 120.4 divided by 4.0?
Operands: [120.4, 4.0]  Operator: /  Final Answer: 30.1
Query: What is 10.5 minus 2.25?
Operands: [10.5, 2.25]  Operator: -  Final Answer: 8.25
